<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/notebooks/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 10 — Semantic Representation Pre-registration & Text Preprocessing**

Phase 9 froze the temporal design **before semantic analysis**: the main trajectory uses 20-year rolling windows with 5-year step (1565–1624), with 25-year and 15-year windows reserved for sensitivity analyses. Phase 10 now freezes the textual and linguistic representation before any network trajectory, distance, change-point, or Renaissance/Baroque comparison is computed.

The central safeguards are:
- Navarro TEI remains the canonical poem-identity backbone and complete primary text layer;
- the Hernández-Lorenzo network corpus is audited only as an independent textual layer and is never mixed silently with Navarro;
- a single reproducible Spanish NLP pipeline is pinned and audited;
- concept definition, vocabulary filtering, co-occurrence context, PPMI rule, and author-balance strategy are fixed **before** Phase 11 builds semantic networks.

**No semantic network, network metric, semantic distance, historical change statistic, or change-point is computed in Phase 10.**


In [ ]:
# Reproducible NLP environment: pin spaCy and the Spanish model wheel.
import sys, subprocess, hashlib, urllib.request
from pathlib import Path

SPACY_VERSION = '3.8.7'
MODEL_NAME = 'es_core_news_sm'
MODEL_VERSION = '3.8.0'
MODEL_URL = 'https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'
MODEL_SHA256 = 'e451a83d6df79b87e9eed0cb553f03e99e36a3bab18a7b79f0dcfd1fdf875e12'
wheel = Path('/content/es_core_news_sm-3.8.0-py3-none-any.whl')
if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest() != MODEL_SHA256:
    urllib.request.urlretrieve(MODEL_URL, wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == MODEL_SHA256
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'spacy=={SPACY_VERSION}', str(wheel)], check=True)
import spacy
assert spacy.__version__ == SPACY_VERSION, spacy.__version__
nlp = spacy.load(MODEL_NAME, disable=['parser','ner'])
assert nlp.meta.get('version') == MODEL_VERSION, nlp.meta
print('NLP pinned:', 'spaCy', spacy.__version__, '|', MODEL_NAME, nlp.meta.get('version'))
print('Model wheel SHA256:', MODEL_SHA256)


In [ ]:
import re, shutil, subprocess, unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
 'hernandez_network':('https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git','ef6b7b691f67abe60d9cfa85c274f0be8095dd9a'),
}
ROOT=Path('/content/gasr_phase10_sources'); ROOT.mkdir(exist_ok=True)
def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst
paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']; HNET=paths['hernandez_network']
XML_ID='{http://www.w3.org/XML/1998/namespace}id'
def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))
rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    author=fp.parent.name; txt='\n'.join(lines)
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'source_file':str(fp.relative_to(N)),'n_lines':len(lines),'lines':lines,'text_tei':txt,'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'first_line':lines[0]})
n=pd.DataFrame(rows); assert len(n)==5078,len(n)
print('Pinned sources ready')
print('Navarro poems:',len(n),'| author folders:',n.author_dir.nunique())


In [ ]:
# Reproduce and freeze the Phase-8 primary chronology (97 poems).
primary_rows=[]
def add(pid,author,lo,hi,confidence,basis):
    primary_rows.append({'n_id':pid,'author_dir':author,'composition_min':int(lo),'composition_max':int(hi),'temporal_confidence':confidence,'temporal_basis':basis})

# Góngora: same pinned scholarly linkage used in Phases 4/5/9.
groot=ET.parse(G/'gongora_obra-poetica.xml').getroot(); parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'scholarly_year':ys[0] if len(ys)==1 else pd.NA,'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False); ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict(); links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy(); collisions=set(pre.g_id.value_counts()[lambda s:s>1].index); glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left'); acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False): add(r.n_id,'Gongora',r.scholarly_year,r.scholarly_year,'A' if r.method=='exact' else 'B','scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link')
phase4_nids=set(acc.n_id); phase4_gids=set(acc.g_id); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy(); first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); recovered=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    if len(ids2)!=1: continue
    gid=ids2[0]
    if gid in phase4_gids: continue
    gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio()
    if score>=0.95 and gr.year_status=='unique' and pd.notna(gr.scholarly_year): recovered.append((r.n_id,gid,score,int(gr.scholarly_year)))
rec=pd.DataFrame(recovered,columns=['n_id','g_id','score','year']); dup=set(rec.g_id.value_counts()[lambda s:s>1].index) if len(rec) else set(); rec=rec[~rec.g_id.isin(dup)]
for r in rec.itertuples(index=False): add(r.n_id,'Gongora',r.year,r.year,'B','scholarly_chronology_year_variant_link')
assert sum(x['author_dir']=='Gongora' for x in primary_rows)==58

GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),35:(1535,1535,'A','historically_anchored_scholarly_year'),**{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items(): add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no:02d}.xml','GarcilasoDeLaVega',lo,hi,conf,basis)
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]: add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),(276,1573,1574,'Bazan_Tunis'),(300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]: add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]: add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_happiness_period_1594_1596')
for no,year,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),(72,1611,'Aminta_1611'),(76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),(43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]: add(f'Quevedo::Quevedo_{no}.xml','Quevedo',year,year,'B',basis)
primary=pd.DataFrame(primary_rows).drop_duplicates('n_id').copy(); expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and primary.groupby('author_dir').size().to_dict()==expected
primary=primary.merge(n[['n_id','text_tei','lines','n_lines','source_file','first_line','signature']],on='n_id',how='left',validate='one_to_one'); assert primary.text_tei.notna().all()
print('PHASE-8 PRIMARY CHRONOLOGY FROZEN: 97 poems')
display(primary.groupby('author_dir').size().sort_values(ascending=False).rename('primary_poems').to_frame())


## Phase 9 temporal design carried forward without re-tuning

The main semantic trajectory is frozen as nine 20-year windows with 5-year step: `1565–1584, 1570–1589, ..., 1605–1624`.

Sensitivity designs are 25-year and 15-year windows. Dates 1580 and 1605 remain external historical reference markers only. Phase 10 may audit text/preprocessing coverage in these windows, but it may not alter the temporal design on the basis of semantic content.


In [ ]:
MAIN_WIDTH=20; MAIN_STEP=5; MAIN_WINDOWS=[(s,s+MAIN_WIDTH-1) for s in range(1565,1606,MAIN_STEP)]
SENSITIVITY_WIDTHS=[25,15]; EXTERNAL_MARKERS=[1580,1605]; MC_DRAWS=1000; SEED=20260825
assert len(MAIN_WINDOWS)==9 and MAIN_WINDOWS[0]==(1565,1584) and MAIN_WINDOWS[-1]==(1605,1624)
print('Temporal design frozen:',MAIN_WINDOWS)


In [ ]:
# Audit Hernández-Lorenzo as a secondary text layer. Exact signature first; unique fuzzy >=.98 second.
H_AUTHOR_FILES={'Gongora':'Gongora_Sonetos.txt','GarcilasoDeLaVega':'Garcilaso_Sonetos.txt','FernandoDeHerrera':'Herrera_Sonetos.txt','Cervantes':'Cervantes_Sonetos.txt','Quevedo':'Quevedo_Sonetos.txt'}
def segment_blocks(path):
    raw=Path(path).read_text(encoding='utf-8',errors='replace').replace('\r\n','\n'); out=[]
    for i,block in enumerate(re.split(r'\n\s*\n+',raw),1):
        lines=[x.strip() for x in block.splitlines() if x.strip()]
        if lines:
            txt='\n'.join(lines); out.append({'h_block_id':i,'h_n_lines':len(lines),'text_standardized':txt,'h_signature':norm(txt)})
    return pd.DataFrame(out)
crosswalk_rows=[]
for author,grp in primary.groupby('author_dir'):
    if author not in H_AUTHOR_FILES:
        for r in grp.itertuples(index=False): crosswalk_rows.append({'n_id':r.n_id,'author_dir':author,'h_block_id':pd.NA,'match_method':'source_unavailable','match_score':pd.NA,'text_standardized':pd.NA})
        continue
    hp=HNET/'corpus'/H_AUTHOR_FILES[author]; assert hp.exists(),hp; hb=segment_blocks(hp); hb14=hb[hb.h_n_lines.eq(14)].copy(); sig_map=hb14.groupby('h_signature').h_block_id.apply(list).to_dict(); prelim=[]
    for r in grp.itertuples(index=False):
        ids=sig_map.get(r.signature,[]); prelim.append((r.n_id,int(ids[0]),'exact',1.0) if len(ids)==1 else (r.n_id,None,'unmatched',np.nan))
    ex=pd.DataFrame(prelim,columns=['n_id','h_block_id','method','score'])
    if ex.h_block_id.notna().any():
        collision_ids=set(ex.loc[ex.h_block_id.notna(),'h_block_id'].value_counts()[lambda s:s>1].index.astype(int)); ex.loc[ex.h_block_id.isin(collision_ids),['h_block_id','method','score']]=[np.nan,'unmatched',np.nan]
    used=set(ex.loc[ex.h_block_id.notna(),'h_block_id'].astype(int)); remaining_h=hb14[~hb14.h_block_id.isin(used)].copy(); fuzzy=[]
    for rr in ex[ex.h_block_id.isna()].itertuples(index=False):
        target_sig=grp.loc[grp.n_id.eq(rr.n_id),'signature'].iloc[0]; best_id,best_score=None,-1.0
        for hr in remaining_h.itertuples(index=False):
            sc=SequenceMatcher(None,target_sig,hr.h_signature).ratio()
            if sc>best_score: best_id,best_score=int(hr.h_block_id),float(sc)
        fuzzy.append((rr.n_id,best_id,best_score))
    fz=pd.DataFrame(fuzzy,columns=['n_id','h_block_id','score'])
    if len(fz):
        fz['preaccept']=fz.score.ge(0.98); fcoll=set(fz.loc[fz.preaccept,'h_block_id'].value_counts()[lambda s:s>1].index); fz['accept']=fz.preaccept&~fz.h_block_id.isin(fcoll)
        for fr in fz[fz.accept].itertuples(index=False): ex.loc[ex.n_id.eq(fr.n_id),['h_block_id','method','score']]=[fr.h_block_id,'fuzzy_ge_0.98',fr.score]
    hb_lookup=hb14.set_index('h_block_id')
    for er in ex.itertuples(index=False):
        if pd.notna(er.h_block_id):
            bid=int(er.h_block_id); crosswalk_rows.append({'n_id':er.n_id,'author_dir':author,'h_block_id':bid,'match_method':er.method,'match_score':float(er.score),'text_standardized':hb_lookup.loc[bid,'text_standardized']})
        else: crosswalk_rows.append({'n_id':er.n_id,'author_dir':author,'h_block_id':pd.NA,'match_method':'unresolved','match_score':pd.NA,'text_standardized':pd.NA})
crosswalk=pd.DataFrame(crosswalk_rows); assert len(crosswalk)==97 and crosswalk.n_id.nunique()==97
primary=primary.merge(crosswalk[['n_id','h_block_id','match_method','match_score','text_standardized']],on='n_id',how='left',validate='one_to_one'); primary['has_standardized']=primary.text_standardized.notna()
coverage_author=primary.groupby('author_dir').agg(primary_poems=('n_id','size'),standardized_matches=('has_standardized','sum')).reset_index(); coverage_author['coverage']=coverage_author.standardized_matches/coverage_author.primary_poems
print('HERNÁNDEZ TEXT-LAYER COVERAGE BY AUTHOR'); display(coverage_author.sort_values('author_dir'))


In [ ]:
# Coverage audit under the already frozen Phase-9 uncertainty model.
rng=np.random.default_rng(SEED); lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int); sampled=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)): sampled[:,j]=a if a==b else rng.integers(a,b+1,size=MC_DRAWS)
has_std=primary.has_standardized.to_numpy(bool); window_cov_rows=[]
for start,end in MAIN_WINDOWS:
    vals=[]; nvals=[]
    for m in range(MC_DRAWS):
        mask=(sampled[m]>=start)&(sampled[m]<=end); nvals.append(int(mask.sum())); vals.append(float(has_std[mask].mean()) if mask.any() else np.nan)
    window_cov_rows.append({'start':start,'end':end,'n_poems_median':float(np.median(nvals)),'standardized_coverage_median':float(np.nanmedian(vals)),'standardized_coverage_q10':float(np.nanquantile(vals,.10)),'standardized_coverage_q90':float(np.nanquantile(vals,.90))})
coverage_window=pd.DataFrame(window_cov_rows)
STD_COMPLETE=bool(primary.has_standardized.all()); MAIN_TEXT_LAYER='text_standardized' if STD_COMPLETE else 'text_tei'; SENSITIVITY_TEXT_LAYER='text_tei' if STD_COMPLETE else 'text_standardized_matched_subset'
assert MAIN_TEXT_LAYER=='text_tei'
print('MAIN TEXT LAYER:',MAIN_TEXT_LAYER); print('SECONDARY TEXT LAYER:',SENSITIVITY_TEXT_LAYER); print('Overall standardized coverage:',round(primary.has_standardized.mean(),3)); display(coverage_window)


## Frozen linguistic representation

The main node unit is **lemma + coarse POS** for content words. Main POS set: `NOUN`, `VERB`, `ADJ`, `ADV`. `PROPN` is excluded from the main representation and reserved for a named-entity-inclusive sensitivity analysis. `AUX`, pronouns, determiners, conjunctions, adpositions and punctuation are excluded by POS rather than by a modern stopword list.

No external lexical stoplist is used in the main pipeline. Technical exclusions are limited to non-alphabetic tokens, empty lemmas, and one-character lemmas.

Because the NLP model is trained on modern Spanish, Phase 10 exports a deterministic quality sample and aggregate `X`/empty-lemma rates. These diagnostics document model risk; they do not permit outcome-driven changes to the historical analysis.


In [ ]:
MAIN_POS={'NOUN','VERB','ADJ','ADV'}; PROPN_SENS_POS=MAIN_POS|{'PROPN'}; MIN_LEMMA_LEN=2
def normalize_lemma(s): return unicodedata.normalize('NFC',str(s)).strip().lower()
line_records=[]
for r in primary.itertuples(index=False):
    for line_no,line in enumerate(r.lines,1): line_records.append((r.n_id,r.author_dir,line_no,line))
docs=list(nlp.pipe([x[3] for x in line_records],batch_size=128)); assert len(docs)==len(line_records); token_rows=[]
for (pid,author,line_no,line),doc in zip(line_records,docs):
    for token_no,tok in enumerate(doc,1):
        lemma=normalize_lemma(tok.lemma_); alpha=bool(tok.is_alpha); technical_ok=alpha and bool(lemma) and len(lemma)>=MIN_LEMMA_LEN; is_main=technical_ok and tok.pos_ in MAIN_POS
        token_rows.append({'n_id':pid,'author_dir':author,'line_no':line_no,'token_no':token_no,'surface':tok.text,'lemma':lemma,'pos':tok.pos_,'concept':f'{lemma}::{tok.pos_}' if technical_ok else '','is_alpha':alpha,'technical_ok':technical_ok,'is_main_content':is_main,'is_propn_sensitivity':technical_ok and tok.pos_ in PROPN_SENS_POS,'spacy_is_stop':bool(tok.is_stop)})
tokens=pd.DataFrame(token_rows); assert len(tokens)>0
poem_audit=tokens.groupby(['n_id','author_dir']).agg(tokens=('surface','size'),alpha_tokens=('is_alpha','sum'),main_content_tokens=('is_main_content','sum'),x_pos_tokens=('pos',lambda s:int((s=='X').sum())),empty_lemma_tokens=('lemma',lambda s:int((s=='').sum()))).reset_index(); poem_audit['main_content_rate']=poem_audit.main_content_tokens/poem_audit.tokens; poem_audit['x_pos_rate']=poem_audit.x_pos_tokens/poem_audit.tokens
author_audit=poem_audit.groupby('author_dir').agg(poems=('n_id','size'),tokens=('tokens','sum'),main_content_tokens=('main_content_tokens','sum'),median_content_rate=('main_content_rate','median'),median_x_pos_rate=('x_pos_rate','median')).reset_index()
quality_pool=tokens[tokens.is_alpha].copy(); quality_sample=quality_pool.sample(n=min(120,len(quality_pool)),random_state=SEED)[['n_id','author_dir','line_no','surface','lemma','pos','is_main_content']].sort_values(['author_dir','n_id','line_no'])
print('NLP TOKEN AUDIT BY AUTHOR'); display(author_audit); print('Total tokens:',len(tokens),'| main content tokens:',int(tokens.is_main_content.sum()),'| X-POS rate:',round(tokens.pos.eq('X').mean(),4))


In [ ]:
# Freeze global vocabulary before any temporal network is built.
MIN_POEM_DF_MAIN=2; MIN_POEM_DF_SENS=3; main_tok=tokens[tokens.is_main_content].copy()
concept_df=main_tok[['n_id','concept','lemma','pos']].drop_duplicates(['n_id','concept']).groupby(['concept','lemma','pos']).n_id.nunique().rename('poem_df').reset_index(); concept_tf=main_tok.groupby('concept').size().rename('token_frequency').reset_index(); vocab=concept_df.merge(concept_tf,on='concept',how='left')
vocab['in_main_vocab_df2']=vocab.poem_df.ge(MIN_POEM_DF_MAIN); vocab['in_sensitivity_vocab_df3']=vocab.poem_df.ge(MIN_POEM_DF_SENS); MAIN_VOCAB=set(vocab.loc[vocab.in_main_vocab_df2,'concept']); SENS_VOCAB=set(vocab.loc[vocab.in_sensitivity_vocab_df3,'concept']); assert len(MAIN_VOCAB)>0
print('Vocabulary frozen before network construction'); print('All content concepts:',len(vocab),'| df>=2 main:',len(MAIN_VOCAB),'| df>=3 sensitivity:',len(SENS_VOCAB))


## Phase 11 network specification frozen in advance

Phase 10 does **not** instantiate these networks; it records the rules Phase 11 must follow.

**Main co-occurrence unit:** one poetic line. Within a line, repeated occurrences of the same concept are collapsed and every unordered pair of distinct retained concepts contributes at most one raw co-occurrence event.

For each temporal window, edge strength will be **PPMI** from line-presence probabilities:

`PPMI(i,j) = max(0, log2[p(i,j)/(p(i)p(j))])`.

The main raw pair-support requirement is at least **2 lines**. Sensitivity checks will use raw support 1 and 3 and an alternative sliding window of 5 retained content tokens.

**Author control:** the raw poem-pooled network is the historical-corpus estimand. A pre-specified author-balanced robustness network will weight each line so each represented author contributes equal total line mass within a temporal window: `w(l,p,a,W)=1/(n[a,W] * L[p])`. Leave-one-author-out is used only where Phase 9 established feasibility.


In [ ]:
config_rows=[('main_temporal_width',20,'years'),('main_temporal_step',5,'years'),('main_text_layer',MAIN_TEXT_LAYER,''),('secondary_text_layer',SENSITIVITY_TEXT_LAYER,''),('nlp_library',f'spacy=={SPACY_VERSION}',''),('nlp_model',f'{MODEL_NAME}=={MODEL_VERSION}',''),('main_pos','NOUN|VERB|ADJ|ADV',''),('proper_nouns_main',False,''),('external_stoplist_main',False,''),('min_lemma_length',MIN_LEMMA_LEN,'characters'),('concept_identity','lemma::coarse_POS',''),('global_min_poem_df_main',MIN_POEM_DF_MAIN,'poems'),('global_min_poem_df_sensitivity',MIN_POEM_DF_SENS,'poems'),('main_cooccurrence_unit','poetic_line_unique_presence',''),('main_edge_weight','PPMI',''),('main_min_raw_pair_support',2,'lines'),('pair_support_sensitivities','1|3','lines'),('alternative_context_sensitivity','5 retained content tokens',''),('author_balance_robustness','equal author total line mass',''),('external_markers','1580|1605','reference_only')]
config=pd.DataFrame(config_rows,columns=['parameter','value','unit_or_role']); display(config)


In [ ]:
OUT=Path('/content/gasr_phase10_outputs'); OUT.mkdir(exist_ok=True); primary_export=primary.drop(columns=['lines'],errors='ignore')
primary_export.to_csv(OUT/'phase10_primary_chronology_and_text_layers.csv',index=False); crosswalk.to_csv(OUT/'phase10_standardized_crosswalk.csv',index=False); coverage_author.to_csv(OUT/'phase10_text_coverage_by_author.csv',index=False); coverage_window.to_csv(OUT/'phase10_text_coverage_by_main_window.csv',index=False); poem_audit.to_csv(OUT/'phase10_nlp_audit_by_poem.csv',index=False); author_audit.to_csv(OUT/'phase10_nlp_audit_by_author.csv',index=False); quality_sample.to_csv(OUT/'phase10_nlp_quality_sample.csv',index=False); vocab.sort_values(['poem_df','token_frequency'],ascending=False).to_csv(OUT/'phase10_global_vocabulary.csv',index=False); config.to_csv(OUT/'phase10_preprocessing_config.csv',index=False)
assert len(primary)==97 and primary.n_id.nunique()==97 and primary.text_tei.notna().all(); assert MAIN_TEXT_LAYER=='text_tei'; assert len(MAIN_WINDOWS)==9; assert set(MAIN_POS)=={'NOUN','VERB','ADJ','ADV'}; assert not tokens.empty and not vocab.empty
print('\nPHASE 10 CHECKPOINT'); print('-------------------'); print('Primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors'); print('Main temporal design:',MAIN_WIDTH,'years | step',MAIN_STEP,'| windows',len(MAIN_WINDOWS)); print('Main text layer:',MAIN_TEXT_LAYER,'| complete coverage:',bool(primary.text_tei.notna().all())); print('Hernández matched text coverage:',int(primary.has_standardized.sum()),'/',len(primary)); print('NLP:',f'spaCy {SPACY_VERSION}','|',f'{MODEL_NAME} {MODEL_VERSION}'); print('Main content POS:',sorted(MAIN_POS)); print('Main global vocabulary df>=2:',len(MAIN_VOCAB)); print('Co-occurrence/PPMI specification frozen: TRUE'); print('Author-balanced robustness specification frozen: TRUE'); print('Semantic networks computed: FALSE'); print('Change-point / historical semantic comparison computed: FALSE'); print('Outputs:',OUT)
